In [1]:
!pip install datasets transformers torch

In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import DataLoader
import torch

#### Lab 1 - LLM Data Pipeline (In-Memory)

**Dataset:** `roneneldan/TinyStories` — a small, modern, script-free dataset of short English stories (~475MB, loads in seconds)  
**Tokenizer:** `roberta-base`

### Pipeline Overview
1. Load TinyStories into memory (small slice of 10,000 stories)
2. Tokenize with RoBERTa tokenizer
3. Group tokens into fixed-length blocks of 128
4. Wrap in a PyTorch DataLoader
5. Sanity-check batch shapes

In [3]:
# ============================================================
# 1. Load TinyStories dataset
# ============================================================
# TinyStories contains short English stories generated by GPT.
# It is stored in modern parquet format — no legacy scripts,
# no trust_remote_code needed, loads in seconds.
#
# We take a slice of 10,000 stories for this in-memory demo.

full_dataset = load_dataset("roneneldan/TinyStories", split="train")
dataset = full_dataset.select(range(10000))

print(f"Number of samples : {len(dataset)}")
print(f"Columns           : {dataset.column_names}")
print(f"\nSample story:\n{dataset[0]['text'][:300]}")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

Number of samples : 10000
Columns           : ['text']

Sample story:
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and 


In [4]:
# ============================================================
# 2. Initialize the RoBERTa tokenizer
# ============================================================
# RoBERTa uses BPE with vocab size 50,265.
# Unlike GPT-2, it has a native <pad> token — no workaround needed.

tokenizer = AutoTokenizer.from_pretrained("roberta-base")

print(f"Vocab size  : {tokenizer.vocab_size}")
print(f"Pad token   : '{tokenizer.pad_token}'  (id={tokenizer.pad_token_id})")
print(f"EOS token   : '{tokenizer.eos_token}'  (id={tokenizer.eos_token_id})")
print(f"BOS token   : '{tokenizer.bos_token}'  (id={tokenizer.bos_token_id})")

Vocab size  : 50265
Pad token   : '<pad>'  (id=1)
EOS token   : '</s>'  (id=2)
BOS token   : '<s>'  (id=0)


In [5]:
# ============================================================
# 3. Tokenize the dataset using batched .map()
# ============================================================
# No padding or truncation — we want raw variable-length token
# sequences so we can concatenate freely across stories.
# add_special_tokens=False keeps the LM stream clean.

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        add_special_tokens=False,
        return_special_tokens_mask=False
    )

tokenized_ds = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing"
)

# Sanity check
sample_ids = tokenized_ds[0]["input_ids"]
print(f"Token IDs (first 20) : {sample_ids[:20]}")
print(f"Decoded              : {tokenizer.decode(sample_ids[:20])}")
print(f"Total tokens in doc 0: {len(sample_ids)}")

Tokenizing:   0%|          | 0/10000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (986 > 512). Running this sequence through the model will result in indexing errors


Token IDs (first 20) : [3762, 183, 6, 10, 410, 1816, 1440, 17990, 303, 10, 19012, 11, 69, 929, 4, 264, 1467, 24, 21, 1202]
Decoded              : One day, a little girl named Lily found a needle in her room. She knew it was difficult
Total tokens in doc 0: 162


In [6]:
# ============================================================
# 4. Group tokens into fixed-length blocks of 128
# ============================================================
# Concatenate ALL token sequences into one long stream,
# then slice into non-overlapping blocks of block_size tokens.
# Tail tokens that don't fill a full block are discarded.

block_size = 128

def group_texts(examples):
    concatenated_input_ids  = sum(examples["input_ids"], [])
    concatenated_attn_masks = sum(examples["attention_mask"], [])

    # Trim to a multiple of block_size
    total_len = (len(concatenated_input_ids) // block_size) * block_size
    concatenated_input_ids  = concatenated_input_ids[:total_len]
    concatenated_attn_masks = concatenated_attn_masks[:total_len]

    # Slice into fixed-length chunks
    result_input_ids  = [concatenated_input_ids[i : i + block_size]  for i in range(0, total_len, block_size)]
    result_attn_masks = [concatenated_attn_masks[i : i + block_size] for i in range(0, total_len, block_size)]

    return {"input_ids": result_input_ids, "attention_mask": result_attn_masks}


lm_ds = tokenized_ds.map(
    group_texts,
    batched=True,
    batch_size=500,
    desc="Grouping into blocks"
)

print(f"Total LM training sequences : {len(lm_ds)}")
print(f"Sequence length check       : {len(lm_ds[0]['input_ids'])} tokens")

Grouping into blocks:   0%|          | 0/10000 [00:00<?, ? examples/s]

Total LM training sequences : 16805
Sequence length check       : 128 tokens


In [7]:
# ============================================================
# 5. Build a PyTorch DataLoader
# ============================================================
# labels = input_ids for causal LM — the model handles the
# internal next-token shift during forward pass.

def collate_fn(batch):
    input_ids      = torch.tensor([ex["input_ids"]      for ex in batch], dtype=torch.long)
    attention_mask = torch.tensor([ex["attention_mask"] for ex in batch], dtype=torch.long)
    return {
        "input_ids"      : input_ids,
        "attention_mask" : attention_mask,
        "labels"         : input_ids.clone()
    }

train_loader = DataLoader(
    lm_ds,
    batch_size=8,
    shuffle=True,
    collate_fn=collate_fn
)

print(f"Number of batches per epoch : {len(train_loader)}")

Number of batches per epoch : 2101


In [8]:
# ============================================================
# 6. Sanity check — inspect a batch
# ============================================================

for batch in train_loader:
    print("input_ids shape      :", batch["input_ids"].shape)       # [8, 128]
    print("attention_mask shape :", batch["attention_mask"].shape)  # [8, 128]
    print("labels shape         :", batch["labels"].shape)          # [8, 128]
    print()
    print("Decoded first sequence (first 100 chars):")
    print(tokenizer.decode(batch["input_ids"][0])[:100])
    break

input_ids shape      : torch.Size([8, 128])
attention_mask shape : torch.Size([8, 128])
labels shape         : torch.Size([8, 128])

Decoded first sequence (first 100 chars):


The little girl smiled and the two of them walked away. They went to the store and bought somethin
